In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import torch
import yaml
from torch.utils.data import DataLoader

In [2]:
def find_project_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(6):
        if (p / "configs" / "config_domainnet.yaml").exists():
            return p
        p = p.parent
    raise FileNotFoundError("Could not find configs/config_domainnet.yaml")


ROOT = find_project_root()
sys.path.insert(0, str(ROOT.parent))

from ebm_unlearning.src.data.domainnet import DomainNetSubset
from ebm_unlearning.src.models.ebm import EnergyModel
from ebm_unlearning.src.training.pretrain import EarlyStoppingConfig, PretrainConfig, pretrain_ebm
from ebm_unlearning.src.utils.logging import setup_logger
from ebm_unlearning.src.utils.seed import set_seed
from ebm_unlearning.src.utils.tracking import make_tracker

with open(ROOT / "configs" / "config_domainnet.yaml", "r") as f:
    cfg = yaml.safe_load(f)

set_seed(int(cfg["seed"]))
device = torch.device(cfg.get("device", "cpu"))

from datetime import datetime

logger = setup_logger("pretrain", log_file=str(ROOT / "outputs" / "logs" / "pretrain_domainnet.log"))
run_id = datetime.now().strftime("%Y%m%d-%H%M%S")
tracker = make_tracker(
    "tensorboard",
    log_dir=str(ROOT / "outputs" / "tensorboard" / "domainnet" / "pretrain" / run_id),
)

dset = DomainNetSubset(
    root=str(ROOT / cfg["data"]["data_dir"]),
    classes=cfg["data"]["classes"],
    domains=cfg["data"]["domains"],
)

batch_size  = int(cfg["data"]["batch_size"])
num_workers = int(cfg["data"]["num_workers"])

val_fraction = float(cfg.get("pretrain", {}).get("val_fraction", 0.1))
val_n   = int(round(len(dset) * val_fraction))
train_n = len(dset) - val_n
train_dset, val_dset = torch.utils.data.random_split(
    dset, [train_n, val_n],
    generator=torch.Generator().manual_seed(int(cfg["seed"])),
)

loader = DataLoader(
    train_dset, batch_size=batch_size, shuffle=True,
    num_workers=num_workers, drop_last=True,
    pin_memory=True, persistent_workers=True,
)
val_loader = DataLoader(
    val_dset, batch_size=16, shuffle=False,
    num_workers=num_workers,
    pin_memory=True, persistent_workers=True,
)

print(f"train/val sizes: {len(train_dset)} / {len(val_dset)}")
print(f"Classes ({len(dset.classes)}): {dset.classes}")

2026-06-02 19:14:42.937411: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-02 19:14:43.381454: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-02 19:14:46.418116: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


[domainnet] 13262 images | 10 classes × 4 domains
train/val sizes: 11936 / 1326
Classes (10): ['airplane', 'bear', 'car', 'dog', 'guitar', 'horse', 'lion', 'tiger', 'truck', 'zebra']


In [3]:
model = EnergyModel(
    in_channels=int(cfg["model"]["in_channels"]),
    hidden_dim=int(cfg["model"]["hidden_dim"]),
    num_classes=int(cfg["model"].get("num_classes", 10)),
    embed_dim=int(cfg["model"].get("embed_dim", 128)),
    backbone=str(cfg["model"].get("backbone", "resnet18")),
    finetune_stages=int(cfg["model"].get("finetune_stages", 2)),
    imagenet_pretrained=bool(cfg["model"].get("imagenet_pretrained", True)),
)

pre_cfg = PretrainConfig(
    epochs=int(cfg["pretrain"]["epochs"]),
    lr=float(cfg["pretrain"]["lr"]),
    weight_decay=float(cfg["pretrain"]["weight_decay"]),
    margin=float(cfg["pretrain"].get("margin", 1.0)),
    k_neg=int(cfg["pretrain"].get("k_neg", 10)),
    neg_chunk=int(cfg["pretrain"].get("neg_chunk", 5)),
    log_every=int(cfg["pretrain"]["log_every"]),
    checkpoint_path=str(ROOT / cfg["pretrain"]["checkpoint_path"]),
)

es_cfg = EarlyStoppingConfig(
    enabled=bool(cfg.get("pretrain", {}).get("early_stopping", {}).get("enabled", True)),
    patience=int(cfg.get("pretrain", {}).get("early_stopping", {}).get("patience", 8)),
    min_delta=float(cfg.get("pretrain", {}).get("early_stopping", {}).get("min_delta", 1e-3)),
    mode=str(cfg.get("pretrain", {}).get("early_stopping", {}).get("mode", "max")),
)

print(f"Model: {sum(p.numel() for p in model.parameters() if p.requires_grad):,} trainable params")
print(f"Checkpoint → {pre_cfg.checkpoint_path}")

Model: 10,560,513 trainable params
Checkpoint → /home/owais/machine unlearning/ebm_unlearning/outputs/checkpoints/ebm_pretrained_domainnet.pt


In [ ]:
model = pretrain_ebm(
    model,
    loader,
    device=device,
    cfg=pre_cfg,
    logger=logger,
    seed=int(cfg["seed"]),
    tracker=tracker,
    val_loader=val_loader,
    early_stopping=es_cfg,
)
tracker.close()